# WIKI LM

Project goal: train an encyclopedic LLM on Wikipedia-style corpora, then fine-tune it into a conversational encyclopedia.

Project workflow:
1. Prepare the dataset (WikiText-2 / WikiText-103) and tokenize it with `tiktoken` or BPE.
2. Build the language-model network.
3. Implement the training algorithm.
4. Train the model.
5. Fine-tune the model.


## Phase 1: Dataset Preparation

Lightweight option: WikiText-2 (suitable for proof-of-concept experiments and fast iteration)

Heavyweight option: WikiText-103 (suitable for more serious training)

No matter what domain-specific task the final language model will perform, basic natural-language ability is indispensable. Based on common practice in the research community, relying only on high-quality domain-specific datasets is far from enough. A broader corpus must be mixed into the training data so that the model can acquire sufficient linguistic knowledge.

### Basic usage of the `datasets` library

- `load_dataset`      Load a dataset from Hugging Face or from local files

- `print`             Inspect the structure: splits, fields, and samples

- `select` / `filter` Sample and filter data

- `map`               Batch-process data

- `train_test_split`  Split a dataset into training and test sets

- `save_to_disk`      Save the dataset locally in the `datasets` format

- `load_from_disk`    Reload a locally saved dataset


In [1]:
from pathlib import Path
from datasets import load_dataset, load_from_disk

local_path = Path("wikitext-103-raw-v1")

# Download the encyclopedic article dataset wikitext-103-raw-v1,
# or load it from local disk if it has already been downloaded.
if local_path.exists():
    print("Loading dataset from local disk")
    ds = load_from_disk(str(local_path))
else:
    print("Local dataset not found. Downloading from Hugging Face...")
    ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")
    ds.save_to_disk(str(local_path))

print(ds)
print(ds["train"][1])
print(ds["train"][3])


/home/isi/weiyu/.pyenv/versions/anaconda3-5.3.1/envs/llm/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading dataset from local disk
DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 1801350
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})
{'text': ' = Valkyria Chronicles III = \n'}
{'text': ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pi

By running `print(ds)`, we can see that the dataset is a `DatasetDict` containing three splits: `'train'`, `'validation'`, and `'test'`. Printing the object also shows basic information about the dataset.

```python
ds                     # The entire dataset dictionary
ds["train"]            # The training split as a table-like dataset
ds["train"].features   # The schema / column structure of the split
ds["train"][0]         # The first row / example in the training split
ds["train"][0]["text"] # The text field in the first row
```

---


### Preprocessing the WikiText Dataset

The raw WikiText structure looks like this:

```python
{"text": ""}
{"text": " = Valkyria Chronicles III = \n"}
{"text": ""}
{"text": " Senjō no Valkyria 3 : Unrecorded Chronicles ..."}
{"text": ""}
{"text": " = = Gameplay = = \n"}
```

WikiText stores articles line by line as separate examples, rather than storing them as complete paragraphs. Titles at different hierarchy levels are saved as independent examples in forms such as:

```python
= Article Title =
= = Section Title = =
= = = Subsection Title = = =
```

Blank lines between paragraphs are also preserved as empty examples.

This fragmented format is not ideal for pre-training, where continuous text is more useful. Therefore, we need to clean and concatenate the dataset first, turning it into a continuous corpus like this:

```python
= Article Title =

Paragraph 1.

= = Section = =

Paragraph 2.
```


In [2]:
from pathlib import Path

out_dir = Path("data/wikitext103_clean_txt")


def normalize_line(text: str) -> str:
    """
    Apply lightweight cleaning to a single WikiText line.
    Note: title structures such as = Title = are not removed.
    """
    text = text.rstrip("\n")
    text = text.rstrip()
    return text


def save_split_to_txt(split_name: str):
    out_file = out_dir / f"{split_name}.txt"

    with out_file.open("w", encoding="utf-8") as f:
        previous_blank = False

        for ex in ds[split_name]:
            line = normalize_line(ex["text"])

            if line.strip() == "":
                if not previous_blank:
                    f.write("\n")
                    previous_blank = True
                continue

            f.write(line + "\n")
            previous_blank = False

    print(f"saved: {out_file}")


if out_dir.exists():
    print(f"Directory {out_dir} already exists. Skipping save.")
else:
    out_dir.mkdir(parents=True, exist_ok=True)

    for split in ["train", "validation", "test"]:
        save_split_to_txt(split)


Directory data/wikitext103_clean_txt already exists. Skipping save.


After obtaining the clean text files, train a dedicated BPE tokenizer on the full corpus, including the `train`, `test`, and `validation` splits.

- Question: Why train a BPE tokenizer specifically for this training corpus?

  - Answer: A corpus-specific tokenizer can better adapt to the special formatting conventions of the data, especially special uses of symbols, such as WikiText title markers like `"= ="` and preprocessing markers such as `@`:

All three regiments of the NK 2nd Division @-@ the 4th , 17th , and 6th , in line from north to south @-@ crossed during the night to the east side of the Naktong River into the 23rd Regiment sector .


In [3]:
from tokenizers import ByteLevelBPETokenizer
from pathlib import Path

paths = [
    "data/wikitext103_clean_txt/train.txt"
]

tokenizer = ByteLevelBPETokenizer()

tokenizer.train(
    files=paths,
    vocab_size=32000,
    min_frequency=2,
    special_tokens=[
        "<s>",
        "<pad>",
        "</s>",
        "<unk>",
        "<mask>",
    ],
)

out_dir = Path("tokenizer-wikitext103-bpe")
out_dir.mkdir(parents=True, exist_ok=True)

tokenizer.save_model(str(out_dir))

print("tokenizer saved")





tokenizer saved


After tokenization, cut the continuous text files into fixed-length token blocks.

Training on fixed-length text chunks improves efficiency, but it also means that the model can only learn context within each block.

```python
Load text
    text_ds = train.txt

Encode the text with the tokenizer
    ids = tokenizer.encode(text_ds)

Slice the token sequence and truncate the remainder
(optional: use overlap)
    block_size = 512
    num_block = len(text_ds) // block_size
    ids = ids[:num_block * block_size]
    i_block = []
    for i in range(0, len(text_ds), block_size):
        i_block.append(text_ds[i:i+block_size])
    label = [x.copy() for x in i_block]

Save the sliced blocks to disk
    save_to_disk(i_block, label)
```


In [4]:
from pathlib import Path
from datasets import Dataset, DatasetDict
from tokenizers import ByteLevelBPETokenizer

block_size = 512

files = {
    "train": "data/wikitext103_clean_txt/train.txt",
    "validation": "data/wikitext103_clean_txt/validation.txt",
    "test": "data/wikitext103_clean_txt/test.txt",
}

tokenizer = ByteLevelBPETokenizer(
    "tokenizer-wikitext103-bpe/vocab.json",
    "tokenizer-wikitext103-bpe/merges.txt",
)


def file_to_blocks(path):
    text = Path(path).read_text(encoding="utf-8")

    ids = tokenizer.encode(text).ids

    # Because labels need to be shifted by one token,
    # we can use at most len(ids) - 1 tokens.
    total_length = len(ids) - 1

    # Truncate to an integer multiple of block_size.
    total_length = (total_length // block_size) * block_size

    input_ids = [
        ids[i:i + block_size]
        for i in range(0, total_length, block_size)
    ]

    labels = [
        ids[i + 1:i + block_size + 1]
        for i in range(0, total_length, block_size)
    ]

    return Dataset.from_dict({
        "input_ids": input_ids,
        "labels": labels,
    })


save_path = Path("data/wikitext103_bpe_lm_block512")

if save_path.exists():
    print(f"Token-block dataset already exists. Loading directly: {save_path}")
    lm_ds = load_from_disk(str(save_path))
else:
    print("Token-block dataset not found. Generating now...")

    lm_ds = DatasetDict({
        split: file_to_blocks(path)
        for split, path in files.items()
    })

    lm_ds.save_to_disk(str(save_path))
    print(f"Token-block dataset saved: {save_path}")

print(lm_ds)
print(lm_ds["train"][0].keys())
print(len(lm_ds["train"][0]["input_ids"]))


Token-block dataset already exists. Loading directly: data/wikitext103_bpe_lm_block512
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 223951
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 469
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 534
    })
})
dict_keys(['input_ids', 'labels'])
512


At this point, dataset preprocessing is complete.

---

## Phase 2: Pre-Training Preparation

Convert the `DatasetDict` into a PyTorch-compatible format, then build the `DataLoader`.


In [5]:
from torch.utils.data import DataLoader

lm_ds.set_format(type="torch", columns=["input_ids", "labels"])

batch_size = 16

train_loader = DataLoader(
    lm_ds["train"],
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    lm_ds["validation"],
    batch_size=batch_size,
    shuffle=False  # The validation set does not need to be shuffled.
)


Build the language model:

- Multi-Head Attention
- Feed Forward
- Layer Norm


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F


class Head(nn.Module):
    """A single-head self-attention layer."""
    def __init__(self, embed_dim, head_size):
        super().__init__()
        self.key = nn.Linear(embed_dim, head_size, bias=False)
        self.query = nn.Linear(embed_dim, head_size, bias=False)
        self.value = nn.Linear(embed_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)

        # Compute attention scores (affinity).
        wei = q @ k.transpose(-2, -1) * (k.shape[-1]**-0.5) # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)

        # Aggregate values.
        v = self.value(x)
        out = wei @ v # (B, T, head_size)
        return out


class MultiHeadAttention(nn.Module):
    """Multiple self-attention heads running in parallel."""
    def __init__(self, embed_dim, num_heads, head_size):
        super().__init__()
        # Build the multi-head attention module.
        # Tip: use nn.ModuleList to store multiple independent Head modules.
        self.heads = nn.ModuleList([Head(embed_dim, head_size) for _ in range(num_heads)])
        # Finally, project the concatenated heads back to the original dimension
        # with a linear layer. This is optional, but recommended.
        self.proj = nn.Linear(num_heads * head_size, embed_dim)

    def forward(self, x):
        # Concatenate the outputs from all heads.
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)

        return out


class FeedForward(nn.Module):
    """A simple linear stack followed by a nonlinear activation function."""
    def __init__(self, embed_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            # Implement the first linear transformation,
            # expanding the dimension from n_embed to 4 * n_embed.
            # Add a ReLU activation function.
            # Implement the second linear transformation,
            # shrinking the dimension back to n_embed.
            # Add dropout after the projection layer. The probability is set to 0.2.
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.ReLU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class LayerNorm(nn.Module):
    """
    A simplified implementation of LayerNorm.

    Note the difference between LayerNorm and BatchNorm:
    BatchNorm normalizes across the batch dimension,
    whereas LayerNorm normalizes across the channel dimension.
    """
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        # Initialize two learnable parameters with nn.Parameter:
        # gamma for scaling and beta for shifting.
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        # Compute the mean and variance of x along the last dimension.
        # Normalize x as (x - mean) / sqrt(var + eps).
        # Apply the affine transformation with gamma and beta.
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)

        x = (x - mean) / torch.sqrt(var + self.eps)
        x = x * self.gamma + self.beta

        return x


class Block(nn.Module):
    """Transformer Block: communication (attention) + computation (FFN)."""

    def __init__(self, n_embed, n_head):
        # n_embed: embedding dimension; n_head: the number of heads we want.
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_embed, n_head, head_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = LayerNorm(n_embed)
        self.ln2 = LayerNorm(n_embed)

    def forward(self, x):
        # Implement residual connections in a Pre-Norm architecture:
        # 1. x = x + Attention(LayerNorm(x))
        # 2. x = x + FFN(LayerNorm(x))
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, block_size, n_embed):
        super().__init__()

        position = torch.arange(block_size).unsqueeze(1)  # (block_size, 1)

        div_term = torch.exp(
            torch.arange(0, n_embed, 2) * (-torch.log(torch.tensor(10000.0)) / n_embed)
        )  # (n_embed / 2,)

        pe = torch.zeros(block_size, n_embed)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # This is not a learnable parameter, but it will be saved with the model
        # and moved to the GPU together with the model.
        self.register_buffer("pe", pe)

    def forward(self, T):
        return self.pe[:T]


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embed, n_head, n_layer, block_size):
        super().__init__()
        # 1. Token embedding table.
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        # 2. Positional encoding, which lets the model know the order of tokens.
        self.position_encoding = SinusoidalPositionalEncoding(block_size, n_embed)
        # 3. Stack multiple Transformer blocks.
        self.blocks = nn.Sequential(*[Block(n_embed, n_head) for _ in range(n_layer)])
        # 4. Final layer normalization.
        self.ln_f = LayerNorm(n_embed)
        # 5. Language-model head, mapping the feature space back to vocabulary size.
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # tok_emb: (B, T, C), pos_emb: (T, C)
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_encoding(T)
        x = tok_emb + pos_emb # Combine content information and positional information.

        x = self.blocks(x)       # Pass through all Transformer blocks.
        x = self.ln_f(x)         # Final normalization.
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.reshape(B * T, C)
            targets = targets.reshape(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # Limit the context length so it never exceeds block_size.
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] # Focus only on the final time step.
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx


---

## Phase 3: Full Training

After all components have been built, instantiate the language model.


In [ ]:
vocab_size = tokenizer.get_vocab_size()
print(f"vocab_size: {vocab_size}")

embed_dim = 512  # embedding dimension
n_head = 8
n_layer = 8  # Number of Transformer blocks.
learning_rate = 3e-4
norm_clip = 1.0
max_iters = 10000

cuda_id = 0
device = torch.device("cuda:{}".format(cuda_id) if torch.cuda.is_available() else "cpu")

gpt_model = GPTLanguageModel(
    vocab_size=vocab_size,
    n_embed=embed_dim,
    n_head=n_head,
    n_layer=n_layer,
    block_size=block_size,
).to(device)

# print(gpt_model)
total_params = sum(p.numel() for p in gpt_model.parameters())
print(f"Number of model parameters: {total_params:,}")


vocab_size: 32000


Select the optimizer.


In [8]:
opt = torch.optim.AdamW(gpt_model.parameters(), 
                        lr=learning_rate, 
                        weight_decay=1e-2,
                        betas=(0.90, 0.95),
                        eps=1e-8)

warmup_iters = 1000
min_lr = learning_rate * 0.1

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    opt,
    start_factor=1e-5,
    end_factor=1.0,
    total_iters=warmup_iters
)

cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt,
    T_max=max_iters - warmup_iters,
    eta_min=min_lr
)

scheduler = torch.optim.lr_scheduler.SequentialLR(
    opt,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_iters]
)

train_iter = iter(train_loader)
val_iter = iter(val_loader)

for i in range(max_iters):
    gpt_model.train()

    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        batch = next(train_iter)

    xb = batch["input_ids"].to(device)
    yb = batch["labels"].to(device)

    logits, loss = gpt_model(xb, yb)

    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(gpt_model.parameters(), norm_clip)
    opt.step()
    scheduler.step()

    if i % 500 == 0:
        print(f"iter {i}: train loss {loss.item():.4f}")

        val_losses = []
        gpt_model.eval()

        with torch.no_grad():
            try:
                val_batch = next(val_iter)
            except StopIteration:
                val_iter = iter(val_loader)
                val_batch = next(val_iter)
            xb, yb = val_batch["input_ids"].to(device), val_batch["labels"].to(device)
            _, loss = gpt_model(xb, yb)
            val_losses.append(loss.item())
        avg_val_loss = sum(val_losses) / len(val_losses)

        print(f"iter {i}: validation loss {avg_val_loss:.4f}")

print("Training complete. Starting evaluation...")
gpt_model.eval()

val_losses = []
with torch.no_grad():
    for batch in val_loader:
        xb, yb = batch["input_ids"], batch["labels"]
        xb, yb = xb.to(device), yb.to(device)
        _, loss = gpt_model(xb, yb)
        val_losses.append(loss.item())
avg_val_loss = sum(val_losses) / len(val_losses)
print(f"Average validation loss: {avg_val_loss:.4f}")

print("Generated sample text:")
context = "The history of natural language processing (NLP) started in the 1950s"
context_ids = torch.tensor(tokenizer.encode(context).ids, dtype=torch.long, device=device).unsqueeze(0)
generated_ids = gpt_model.generate(context_ids, max_new_tokens=100)[0].tolist()
generated_text = tokenizer.decode(generated_ids)
print(generated_text)


iter 0: train loss 10.5260
iter 0: validation loss 10.5144
iter 500: train loss 6.6352
iter 500: validation loss 6.4972
iter 1000: train loss 6.1487
iter 1000: validation loss 6.1278
iter 1500: train loss 5.8121
iter 1500: validation loss 5.5403
iter 2000: train loss 5.4794
iter 2000: validation loss 5.3177
iter 2500: train loss 5.1524
iter 2500: validation loss 5.1840
iter 3000: train loss 5.3038
iter 3000: validation loss 4.9507
iter 3500: train loss 5.2496
iter 3500: validation loss 4.8567
iter 4000: train loss 4.9541
iter 4000: validation loss 4.7206
iter 4500: train loss 4.7521
iter 4500: validation loss 4.9549
iter 5000: train loss 4.9330
iter 5000: validation loss 4.7767
iter 5500: train loss 4.7075
iter 5500: validation loss 4.6654
iter 6000: train loss 4.7829
iter 6000: validation loss 4.5665
iter 6500: train loss 4.8599
iter 6500: validation loss 4.1958
iter 7000: train loss 4.5636
iter 7000: validation loss 4.6611
iter 7500: train loss 4.5837
iter 7500: validation loss 4.678

In [13]:
import math

batch_size = 16

test_loader = DataLoader(
    lm_ds["test"],
    batch_size=batch_size,
    shuffle=False
)
test_iter = iter(test_loader)

@torch.no_grad()
def evaluate_metrics(model, data_iter, num_batches=50):
    model.eval()
    total_loss = 0
    total_tokens = 0

    for _ in range(num_batches):
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(test_loader)
            batch = next(data_iter)

        xb = batch["input_ids"].to(device)
        yb = batch["labels"].to(device)

        _, loss = model(xb, yb)

        total_loss += loss.item() * batch_size * block_size
        total_tokens += batch_size * block_size

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)

    model.train()

    return {
        "Loss": avg_loss,
        "Perplexity": perplexity
    }


test_ids = []

for item in lm_ds["test"]:
    test_ids.extend(item["input_ids"])

metrics = evaluate_metrics(gpt_model, test_iter)

print("Test-set evaluation results:")
print(f"- Cross Entropy Loss: {metrics['Loss']:.4f}")
print(f"- Perplexity: {metrics['Perplexity']:.2f}")


Test-set evaluation results:
- Cross Entropy Loss: 3.5393
- Perplexity: 34.44


After training is complete, save the checkpoint for later SFT use.


In [16]:
# =========================
# Save model checkpoint
# =========================
import os

save_dir = "model/gpt_wikitext103_layer10"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "checkpoint.pt")

total_params = sum(p.numel() for p in gpt_model.parameters())

checkpoint = {
    "model_state_dict": gpt_model.state_dict(),
    "optimizer_state_dict": opt.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),

    "vocab_size": vocab_size,
    "embed_dim": embed_dim,
    "n_head": n_head,
    "n_layer": n_layer,
    "block_size": block_size,

    "learning_rate": learning_rate,
    "norm_clip": norm_clip,
    "max_iters": max_iters,
    "warmup_iters": warmup_iters,
    "min_lr": min_lr,

    "total_params": total_params,
    "test_loss": metrics["Loss"],
    "test_perplexity": metrics["Perplexity"],
}

torch.save(checkpoint, save_path)

print(f"checkpoint saved to: {save_path}")
print(f"Number of model parameters: {total_params:,}")


checkpoint saved to: model/gpt_wikitext103_layer10/checkpoint.pt
Number of model parameters: 64,309,504
